# Praktikum Mandiri Latihan 3: Prediksi Cuaca dengan KNN

## Deskripsi
Notebook ini menggunakan algoritma K-Nearest Neighbors (KNN) untuk memprediksi tipe cuaca berdasarkan berbagai fitur meteorologi.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/weather_classification_data.csv')

# Tampilkan informasi dasar
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nStatistical Summary:")
print(df.describe())

## 3. Eksplorasi Data

In [ ]:
# Cek missing values
print("Missing Values:")
print(df.isnull().sum())

# Cek distribusi target variable
print("\n\nDistribusi Weather Type:")
print(df['Weather Type'].value_counts())

# Visualisasi distribusi weather type
plt.figure(figsize=(10, 6))
df['Weather Type'].value_counts().plot(kind='bar', color='skyblue')
plt.title('Distribusi Tipe Cuaca', fontsize=14, fontweight='bold')
plt.xlabel('Weather Type')
plt.ylabel('Jumlah')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi korelasi fitur numerik
plt.figure(figsize=(12, 8))
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Encode categorical features
label_encoders = {}
categorical_cols = ['Cloud Cover', 'Season', 'Location']

df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"\n{col} encoding:")
    for i, label in enumerate(le.classes_):
        print(f"  {label}: {i}")

# Encode target variable
le_target = LabelEncoder()
df_encoded['Weather Type'] = le_target.fit_transform(df['Weather Type'])
print("\n\nWeather Type encoding:")
for i, label in enumerate(le_target.classes_):
    print(f"  {label}: {i}")

print("\n\nEncoded Data Shape:", df_encoded.shape)
print(df_encoded.head())

In [ ]:
# Split features and target
X = df_encoded.drop('Weather Type', axis=1)
y = df_encoded['Weather Type']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

In [ ]:
# Split data menjadi training dan testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("\nTraining target distribution:")
print(pd.Series(y_train).value_counts())
print("\nTesting target distribution:")
print(pd.Series(y_test).value_counts())

In [ ]:
# Feature Scaling - KNN sensitive to feature scales
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Original Training data sample:")
print(X_train.head())
print("\n\nScaled Training data sample:")
print(pd.DataFrame(X_train_scaled, columns=X_train.columns).head())

## 5. Mencari Nilai K Optimal

In [ ]:
# Testing berbagai nilai K untuk mencari yang optimal
k_range = range(1, 31)
train_scores = []
test_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_score = knn.score(X_train_scaled, y_train)
    test_score = knn.score(X_test_scaled, y_test)
    
    train_scores.append(train_score)
    test_scores.append(test_score)
    
    if k % 5 == 1:
        print(f"K={k}: Train Accuracy={train_score:.4f}, Test Accuracy={test_score:.4f}")

# Visualisasi hasil
plt.figure(figsize=(12, 6))
plt.plot(k_range, train_scores, label='Training Accuracy', marker='o', linewidth=2)
plt.plot(k_range, test_scores, label='Testing Accuracy', marker='s', linewidth=2)
plt.xlabel('K Value', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN: Accuracy vs K Value', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Cari K optimal
optimal_k = k_range[test_scores.index(max(test_scores))]
print(f"\n\nOptimal K Value: {optimal_k}")
print(f"Best Test Accuracy: {max(test_scores):.4f}")

## 6. Training Model KNN dengan K Optimal

In [ ]:
# Training model dengan K optimal
knn_model = KNeighborsClassifier(n_neighbors=optimal_k)
knn_model.fit(X_train_scaled, y_train)

# Prediksi
y_pred_train = knn_model.predict(X_train_scaled)
y_pred_test = knn_model.predict(X_test_scaled)

# Evaluasi
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Model KNN dengan K={optimal_k}")
print("="*50)
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("="*50)

## 7. Evaluasi Model

In [ ]:
# Classification Report
print("Classification Report (Test Set):")
print("="*70)
print(classification_report(y_test, y_pred_test, target_names=le_target.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le_target.classes_, 
            yticklabels=le_target.classes_,
            cbar_kws={"shrink": 0.8})
plt.title('Confusion Matrix - KNN Weather Prediction', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Cross-Validation

In [ ]:
# Perform cross-validation
cv_scores = cross_val_score(knn_model, X_train_scaled, y_train, cv=5)

print("Cross-Validation Results (5-Fold):")
print("="*50)
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.4f}")
print("="*50)
print(f"Mean CV Score: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

# Visualisasi CV scores
plt.figure(figsize=(10, 6))
plt.bar(range(1, 6), cv_scores, color='skyblue', edgecolor='navy', alpha=0.7)
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.title('Cross-Validation Scores', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Hyperparameter Tuning dengan GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'n_neighbors': range(3, 21, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Perform Grid Search
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, 
                           scoring='accuracy', n_jobs=-1, verbose=1)

print("Performing Grid Search...")
grid_search.fit(X_train_scaled, y_train)

# Results
print("\n\nBest Parameters:", grid_search.best_params_)
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Test best model
best_knn = grid_search.best_estimator_
y_pred_best = best_knn.predict(X_test_scaled)
best_test_accuracy = accuracy_score(y_test, y_pred_best)

print(f"\nBest Model Test Accuracy: {best_test_accuracy:.4f}")

## 10. Prediksi dengan Data Baru

In [ ]:
# Contoh prediksi dengan data baru
# Format: Temperature, Humidity, Wind Speed, Precipitation (%), Cloud Cover, 
#         Atmospheric Pressure, UV Index, Season, Visibility (km), Location

# Contoh 1: Kondisi cuaca cerah
data_baru_1 = {
    'Temperature': 28,
    'Humidity': 45,
    'Wind Speed': 5.0,
    'Precipitation (%)': 10,
    'Cloud Cover': 'clear',
    'Atmospheric Pressure': 1020.0,
    'UV Index': 8,
    'Season': 'Summer',
    'Visibility (km)': 10.0,
    'Location': 'coastal'
}

# Contoh 2: Kondisi cuaca hujan
data_baru_2 = {
    'Temperature': 15,
    'Humidity': 85,
    'Wind Speed': 12.0,
    'Precipitation (%)': 80,
    'Cloud Cover': 'overcast',
    'Atmospheric Pressure': 995.0,
    'UV Index': 2,
    'Season': 'Winter',
    'Visibility (km)': 3.0,
    'Location': 'inland'
}

# Contoh 3: Kondisi cuaca bersalju
data_baru_3 = {
    'Temperature': -5,
    'Humidity': 90,
    'Wind Speed': 15.0,
    'Precipitation (%)': 85,
    'Cloud Cover': 'overcast',
    'Atmospheric Pressure': 985.0,
    'UV Index': 1,
    'Season': 'Winter',
    'Visibility (km)': 2.0,
    'Location': 'mountain'
}

# Fungsi untuk melakukan prediksi
def predict_weather(data_dict):
    # Convert to DataFrame
    df_new = pd.DataFrame([data_dict])
    
    # Encode categorical features
    for col in categorical_cols:
        df_new[col] = label_encoders[col].transform(df_new[col])
    
    # Reorder columns to match training data
    df_new = df_new[X.columns]
    
    # Scale features
    df_new_scaled = scaler.transform(df_new)
    
    # Predict
    prediction = best_knn.predict(df_new_scaled)
    prediction_proba = best_knn.predict_proba(df_new_scaled)
    
    # Decode prediction
    weather_type = le_target.inverse_transform(prediction)[0]
    
    return weather_type, prediction_proba[0]

# Prediksi untuk setiap contoh
print("Prediksi Cuaca dengan Model KNN")
print("="*70)

for i, data in enumerate([data_baru_1, data_baru_2, data_baru_3], 1):
    print(f"\nContoh {i}:")
    print(f"  Temperature: {data['Temperature']}°C")
    print(f"  Humidity: {data['Humidity']}%")
    print(f"  Wind Speed: {data['Wind Speed']} km/h")
    print(f"  Precipitation: {data['Precipitation (%)']}%")
    print(f"  Cloud Cover: {data['Cloud Cover']}")
    
    weather, proba = predict_weather(data)
    print(f"\n  Prediksi: {weather}")
    print(f"  Probabilitas:")
    for j, class_name in enumerate(le_target.classes_):
        print(f"    {class_name}: {proba[j]:.4f}")
    print("-"*70)

## 11. Simpan Model

In [ ]:
import pickle

# Simpan model, scaler, dan encoders
model_data = {
    'model': best_knn,
    'scaler': scaler,
    'label_encoders': label_encoders,
    'target_encoder': le_target,
    'feature_columns': X.columns.tolist(),
    'best_params': grid_search.best_params_,
    'test_accuracy': best_test_accuracy
}

# Save to pickle file
model_path = '../model/knn_weather_prediction_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Model berhasil disimpan di: {model_path}")
print(f"\nModel Information:")
print(f"  - Algorithm: K-Nearest Neighbors")
print(f"  - Best Parameters: {grid_search.best_params_}")
print(f"  - Test Accuracy: {best_test_accuracy:.4f}")
print(f"  - Number of Features: {len(X.columns)}")
print(f"  - Weather Types: {list(le_target.classes_)}")

## Kesimpulan

Model K-Nearest Neighbors (KNN) telah berhasil dibangun untuk prediksi cuaca dengan fitur-fitur berikut:

### Tahapan yang Dilakukan:
1. **Load dan Eksplorasi Data** - Memuat dataset weather classification dan menganalisis strukturnya
2. **Data Preprocessing** - Encoding fitur kategorikal dan normalisasi fitur numerik
3. **Pencarian K Optimal** - Testing berbagai nilai K untuk menemukan yang terbaik
4. **Training Model** - Melatih model KNN dengan parameter optimal
5. **Evaluasi Model** - Menggunakan accuracy, confusion matrix, dan classification report
6. **Cross-Validation** - Validasi performa model dengan 5-fold CV
7. **Hyperparameter Tuning** - GridSearchCV untuk mencari kombinasi parameter terbaik
8. **Prediksi Data Baru** - Mengimplementasikan fungsi prediksi untuk data baru
9. **Simpan Model** - Menyimpan model dan komponen terkait untuk penggunaan di masa depan

### Kelebihan KNN untuk Prediksi Cuaca:
- **Simple dan Intuitif** - Mudah dipahami dan diimplementasikan
- **Non-parametric** - Tidak membuat asumsi tentang distribusi data
- **Adaptif** - Dapat menangkap pola kompleks dalam data cuaca

### Catatan:
- Feature scaling sangat penting karena KNN sensitif terhadap skala fitur
- Nilai K optimal dipilih berdasarkan testing accuracy
- Model disimpan dengan semua komponen preprocessing untuk kemudahan deployment